In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
print(df.shape)
print(df.head())

In [ ]:
col = ["PassengerId", "Name", "Ticket", "Cabin"]
df = df.drop(columns = col)

In [ ]:
print(df.info())

In [ ]:
df['Pclass'] = df["Pclass"].astype(str)

In [ ]:
df['Sex'] = df['Sex'].map({"male": 0, "female": 1})

In [ ]:
print(df['Embarked'].value_counts())

In [ ]:
print(df['Survived'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print(X_train.isnull().sum())

In [ ]:
age_median = X_train["Age"].median()
X_train["Age"] = X_train["Age"].fillna(age_median)
X_test["Age"] = X_test["Age"].fillna(age_median)

In [ ]:
emb = X_train["Embarked"].mode()[0]
X_train["Embarked"] = X_train["Embarked"].fillna(age_median)
X_test["Embarked"] = X_test["Embarked"].fillna(age_median)

In [ ]:
viz_df = pd.concat([X_train, y_train],axis = 1)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

corr = viz_df.select_dtypes(include='number').corr()
sns.heatmap(corr, annot = True, cmap = 'coolwarm')
plt.title("Coorelation Heatmap")

In [ ]:
sns.histplot(data = viz_df, x = 'Age', hue = "Sex", bins = 20, kde = True)
plt.show()

In [ ]:
X_train["Sex"] = X_train["Sex"].astype(str)
X_test['Sex'] = X_test['Sex'].astype(str)
X_train['Embarked'] = X_train['Embarked'].astype(str)
X_test['Embarked'] = X_test['Embarked'].astype(str)

In [ ]:
numcols = X_train.select_dtypes(include = 'number').columns
print(numcols)
strcols = X_train.select_dtypes(include = 'str').columns
print(strcols)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numcols] = scaler.fit_transform(X_train_scaled[numcols])
X_test_scaled[numcols] = scaler.transform(X_test_scaled[numcols])

In [ ]:
X_train_scaled = pd.get_dummies(X_train_scaled, columns= strcols, drop_first = True)
X_test_scaled = pd.get_dummies(X_test_scaled, columns=strcols, drop_first=True)

X_train_scaled, X_test_scaled = X_train_scaled.align(X_test_scaled, join = 'left', axis = 1, fill_value=0)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000, class_weight= "balanced" ),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight="balanced")
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    print(f"{name}: Test: {test_acc} | Train: {train_acc}")

In [ ]:
from sklearn.metrics import classification_report

for name, model in models.items():
    print(name)
    print(classification_report(y_test, model.predict(X_test_scaled)))

In [ ]:
from sklearn.model_selection import cross_val_score

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv = 5)
    print(f"{name}: {scores.mean()} +/- {scores.std()}")

In [ ]:
rf_param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2,5]   
}

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced"),
    rf_param_grid,
    cv = 5,
    n_jobs = -1,
    scoring = 'accuracy'
)

grid.fit(X_train_scaled, y_train)
print(grid.best_params_)
print(grid.best_score_)
best_model =  grid.best_estimator_
print(best_model.score(X_test_scaled, y_test)) 

In [ ]:
importances = pd.Series(best_model.feature_importances_, index = X_train_scaled.columns)
print(importances.sort_values())

In [ ]:
import joblib

joblib.dump(best_model, "Titanic.pkl")

In [ ]:
loaded_model = joblib.load("Titanic.pkl")
print(loaded_model.score(X_test_scaled, y_test))

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = loaded_model.predict(X_test_scaled)
cm =  confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot = True, fmt = 'd', xticklabels=["Died", "Survived"], yticklabels=["Died", "Survived"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
print(classification_report(y_test, best_model.predict(X_test_scaled)))